# Name: [Asha Kilaru]
**Course:** MSCS 634 - Data Visualization & Preprocessing
**Lab:** Lab 1 — Data Visualization, Preprocessing, and Statistical Analysis

This notebook demonstrates data exploration, visualization, preprocessing, and basic statistical analysis on the `tips` dataset (restaurant tips). Run all cells to reproduce figures and save screenshots into the `screenshots/` folder.

In [ ]:
# Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Ensure output directories
os.makedirs('data', exist_ok=True)
os.makedirs('screenshots', exist_ok=True)
plt.style.use('seaborn')

In [ ]:
# Load the dataset. If local CSV exists, load it; otherwise load from seaborn and save a local copy.
data_path = 'data/tips.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    df = sns.load_dataset('tips')
    df.to_csv(data_path, index=False)

# Display the first five rows (screenshot required)
display(df.head())
# Save head snapshot for inclusion in screenshots folder
df.head().to_csv('screenshots/df_head_before.csv', index=False)

## Step 2: Data Visualization
Below are several visualizations. Each figure is saved to `screenshots/` so you can include them in your lab submission.

In [ ]:
# Scatter plot: total_bill vs tip
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='total_bill', y='tip', hue='time')
plt.title('Total Bill vs Tip by Time of Day')
plt.tight_layout()
plt.savefig('screenshots/scatter_totalbill_tip.png')
plt.show()

In [ ]:
# Histogram: distribution of total_bill
plt.figure(figsize=(8,5))
sns.histplot(df['total_bill'], bins=20, kde=True)
plt.title('Distribution of Total Bill')
plt.tight_layout()
plt.savefig('screenshots/hist_total_bill.png')
plt.show()

In [ ]:
# Box plot: tips by day to identify outliers
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='day', y='tip')
plt.title('Tip Distribution by Day')
plt.tight_layout()
plt.savefig('screenshots/boxplot_tip_by_day.png')
plt.show()

## Step 3: Data Preprocessing
We detect missing values, demonstrate handling techniques, detect and remove outliers using IQR, perform data reduction (sampling and column drop), and apply scaling and discretization. Each step saves before/after snapshots to `screenshots/`.

In [ ]:
# 3.1 Handling missing values
# Report missing values in the original dataset
missing_before = df.isnull().sum()
print('Missing values before handling:
', missing_before)
missing_before.to_csv('screenshots/missing_before.csv')
# For demonstration, create a copy and introduce some missing values artificially
df_mv = df.copy()
np.random.seed(0)
# introduce NaNs in 2% of total_bill and tip values
mask = np.random.rand(len(df_mv)) < 0.02
df_mv.loc[mask, 'total_bill'] = np.nan
mask2 = np.random.rand(len(df_mv)) < 0.02
df_mv.loc[mask2, 'tip'] = np.nan
# Save a snapshot before handling
df_mv.head(10).to_csv('screenshots/df_with_missing_before.csv', index=False)
# Handle missing: fill numeric with mean, categorical with mode
df_mv['total_bill'].fillna(df_mv['total_bill'].mean(), inplace=True)
df_mv['tip'].fillna(df_mv['tip'].mean(), inplace=True)
# Save after handling
df_mv.head(10).to_csv('screenshots/df_with_missing_after.csv', index=False)
print('Applied mean imputation to artificially introduced NaNs.')

In [ ]:
# 3.2 Outlier detection and removal using IQR for 'total_bill'
Q1 = df['total_bill'].quantile(0.25)
Q3 = df['total_bill'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['total_bill'] < lower) | (df['total_bill'] > upper)]
print(f'Found {len(outliers)} outliers for total_bill using IQR.
')
# Save IQR and outliers list
with open('screenshots/iqr_info.txt', 'w') as f:
    f.write(f'Q1={Q1}
Q3={Q3}
IQR={IQR}
lower={lower}
upper={upper}
')
outliers.to_csv('screenshots/outliers_total_bill.csv', index=False)
# Remove outliers to produce a cleaned dataset
df_no_out = df[(df['total_bill'] >= lower) & (df['total_bill'] <= upper)].copy()
df_no_out.to_csv('screenshots/df_after_outlier_removal.csv', index=False)
print('Saved outlier list and dataset after outlier removal.')

In [ ]:
# 3.3 Data reduction: sampling and dropping a less relevant column
df_reduced_before = df.copy()
df_reduced_before.to_csv('screenshots/df_before_reduction.csv', index=False)
# Sample 50% of rows
df_sampled = df.sample(frac=0.5, random_state=1).reset_index(drop=True)
# Drop the 'sex' column as a demonstration of dimensionality reduction
if 'sex' in df_sampled.columns:
    df_sampled_drop = df_sampled.drop(columns=['sex'])
else:
    df_sampled_drop = df_sampled
df_sampled_drop.to_csv('screenshots/df_after_reduction.csv', index=False)
print('Saved dataset before and after reduction (sampling + column drop).')

In [ ]:
# 3.4 Scaling and discretization
df_scale = df[['total_bill', 'tip']].copy()
# Save before scaling
df_scale.head().to_csv('screenshots/df_before_scaling.csv', index=False)
# Min-Max scaling
scaler = MinMaxScaler()
df_scale[['total_bill_scaled','tip_scaled']] = scaler.fit_transform(df_scale[['total_bill','tip']])
df_scale.head().to_csv('screenshots/df_after_scaling.csv', index=False)
# Discretize total_bill into 3 bins (Low/Medium/High)
df['total_bill_cat'] = pd.cut(df['total_bill'], bins=3, labels=['Low','Medium','High'])
df[['total_bill','total_bill_cat']].head().to_csv('screenshots/df_after_discretization.csv', index=False)
print('Saved scaling and discretization snapshots.')

## Step 4: Statistical Analysis
Compute general overview, central tendency, dispersion, and correlation matrix. Save textual outputs to `screenshots/`.

In [ ]:
# 4.1 General overview: info and describe
with open('screenshots/data_info.txt', 'w') as f:
    df.info(buf=f)
# describe()
desc = df.describe(include='all')
desc.to_csv('screenshots/describe_all.csv')
print('Saved .info() and .describe() outputs.')

In [ ]:
# 4.2 Central tendency measures (min, max, mean, median, mode) for numeric columns
central = {}
for col in df.select_dtypes(include=[np.number]).columns:
    central[col] = {
        'min': float(df[col].min()),
        'max': float(df[col].max()),
        'mean': float(df[col].mean()),
        'median': float(df[col].median()),
        'mode': list(df[col].mode().values)
    }
pd.DataFrame(central).to_csv('screenshots/central_tendency.csv')
print('Saved central tendency measures.')

In [ ]:
# 4.3 Dispersion measures: range, quartiles, IQR, variance, std dev
disp = {}
for col in df.select_dtypes(include=[np.number]).columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    disp[col] = {
        'range': float(df[col].max() - df[col].min()),
        'q1': float(q1),
        'q3': float(q3),
        'iqr': float(iqr),
        'variance': float(df[col].var()),
        'std_dev': float(df[col].std())
    }
pd.DataFrame(disp).to_csv('screenshots/dispersion_measures.csv')
print('Saved dispersion measures.')

In [ ]:
# 4.4 Correlation analysis
corr = df.select_dtypes(include=[np.number]).corr()
corr.to_csv('screenshots/correlation_matrix.csv')
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix (numeric columns)')
plt.tight_layout()
plt.savefig('screenshots/correlation_heatmap.png')
plt.show()
print('Saved correlation matrix and heatmap.')

---
Run the notebook from top to bottom. The code saves figures and CSV/text snapshots into the `screenshots/` directory which you can include in your repository as required by the assignment.
**Next steps for submission:**
- Run all cells to generate the `data/` and `screenshots/` contents.
- Add `MSCS_634_Lab_1.ipynb`, the `data/` CSV, and the `screenshots/` folder to your GitHub repository.